In [2]:
# Install PyTorch
import torch
print(f"PyTorch version: {torch.__version__}")

import torch.nn as nn

PyTorch version: 2.9.1


## Self Attention Weights

Given: the input text has been arranged into input vectors by the embedding layer.
Objective: Compute context vector as the combination of all input vectors and attention weights. The weights defines the importance of input tokens relative to the target query token.

Intermediate attention scores (omega) are computed by the dot product of an input query with each input token. The dot product is a measure of similarity between two vectors. Attention scores are not normalized.

The attention weights (alpha) are computed by normalizing the attention scores. The normalized values sum to one and helps prevent large values during optimization.

Attention weights (alpha) are multiplied to the input vecctor by dot product.

In [3]:
# Test input vector
inputs = torch.tensor([
  [0.43, 0.15, 0.89], # Your     (x^1
  [0.55, 0.87, 0.66], # journey  (x^2)
  [0.57, 0.85, 0.64], # starts   (x^3)
  [0.22, 0.58, 0.33], # with     (x^4)
  [0.77, 0.25, 0.10], # one      (x^5)
  [0.05, 0.80, 0.55]  # step     (x^6)
])

In [ ]:
# Make the "journey" input token the query
query = inputs[1]

# Compute attention scores by dot product with query for each input 
attn_scores_2 = torch.empty(inputs.shape[0])
for i, x_i in enumerate(inputs):
  attn_scores_2[i] = torch.dot(x_i, query)
print(attn_scores_2)

# Compute attention weights by normalizing the attention scores
attn_weights_2_tmp = attn_scores_2 / attn_scores_2.sum()
print("Attention weights:", attn_weights_2_tmp)
print("Weights sum:", attn_weights_2_tmp.sum())

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])
Attention weights: tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])
Sum: tensor(1.0000)


In [9]:
# Improved normalization using softmax
attn_weights_2 = torch.softmax(attn_scores_2, dim=0)
print("Attention weights:", attn_weights_2)
print("Weights sum:", attn_weights_2.sum())

Attention weights: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Weights sum: tensor(1.)


In [15]:
# Make the "journey" input token the query
query = inputs[1]

# Compute context vector by multiplying by each input 
context_vec_2 = torch.zeros(query.shape)
for i, x_i in enumerate(inputs):
  context_vec_2 += attn_weights_2[i] * x_i
print(context_vec_2)

tensor([0.4419, 0.6515, 0.5683])


Generalize the self-attention mechanism

In [ ]:
# Compute attenion scores by matrix multiplication 
attn_scores = inputs @ inputs.T

# Compute attention weights by normalization
# In softmax() the dim=1 means all row values sum to 1
attn_weights = torch.softmax(attn_scores, dim=1)
print(attn_weights)

# Check all rows sum to 1
attn_weights.sum(dim=-1)

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])


tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000])

In [22]:
# Compute context vectors
compute_vec = attn_weights @ inputs
print(compute_vec)

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])


## Trainable Attention Weights

Trainable weight matrices are divided into *query*, *key* and *value* that are applied to specific input vectors to create *query*, *key* and *value* vectors.

In [31]:
# Trainable self-attention computes a context
class SelfAttention_v2(nn.Module):
  def __init__(self, d_in, d_out, qkt_bias=False):
    super().__init__()

    # Initialize the random trainable weight matrices
    self.W_query = nn.Linear(d_in, d_out, bias=qkt_bias)
    self.W_key = nn.Linear(d_in, d_out, bias=qkt_bias)
    self.W_value = nn.Linear(d_in, d_out, bias=qkt_bias)

  # Forward pass
  def forward(self, x):
    # Compute weight vectors
    queries = self.W_query(x)
    keys = self.W_key(x)
    values = self.W_value(x)

    # Compute attention scores
    attn_scores = queries @ keys.T # omega

    # Compute attention weights by normalizing with softmax
    attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)

    # Compute context vectors of all inputs
    context_vec = attn_weights @ values
    return context_vec

In [32]:
# Test SelfAttention_v1 class
d_in = inputs.shape[1]
d_out = 2
torch.manual_seed(123)
sa_v1 = SelfAttention_v2(d_in, d_out)
print(sa_v1(inputs))

tensor([[-0.5337, -0.1051],
        [-0.5323, -0.1080],
        [-0.5323, -0.1079],
        [-0.5297, -0.1076],
        [-0.5311, -0.1066],
        [-0.5299, -0.1081]], grad_fn=<MmBackward0>)


## Causal Attention

Change the attention weights to ignore future input text which should be hidden during training.

In [41]:
# Test triangular matrix masks
triu_mask = torch.triu(torch.ones(5, 5), diagonal=1)
print(triu_mask)
triu_mask = triu_mask.masked_fill(triu_mask.bool(), -torch.inf)
print(triu_mask)

tril_mask = torch.tril(torch.ones(5, 5))
print(tril_mask)
tril_mask = tril_mask.masked_fill(tril_mask.bool(), -torch.inf)
print(tril_mask)

tensor([[0., 1., 1., 1., 1.],
        [0., 0., 1., 1., 1.],
        [0., 0., 0., 1., 1.],
        [0., 0., 0., 0., 1.],
        [0., 0., 0., 0., 0.]])
tensor([[0., -inf, -inf, -inf, -inf],
        [0., 0., -inf, -inf, -inf],
        [0., 0., 0., -inf, -inf],
        [0., 0., 0., 0., -inf],
        [0., 0., 0., 0., 0.]])
tensor([[1., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0.],
        [1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1.]])
tensor([[-inf, 0., 0., 0., 0.],
        [-inf, -inf, 0., 0., 0.],
        [-inf, -inf, -inf, 0., 0.],
        [-inf, -inf, -inf, -inf, 0.],
        [-inf, -inf, -inf, -inf, -inf]])


In [ ]:
# Causal self-attention computes a context
class CausalAttention(nn.Module):
  def __init__(self, d_in, d_out, context_length, dropout, qkt_bias=False):
    super().__init__()
    self.d_out = d_out

    # Initialize the random trainable weight matrices
    self.W_query = nn.Linear(d_in, d_out, bias=qkt_bias)
    self.W_key = nn.Linear(d_in, d_out, bias=qkt_bias)
    self.W_value = nn.Linear(d_in, d_out, bias=qkt_bias)

    # Set dropout
    self.dropout = nn.Dropout(dropout)

    #
    self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1))


  # TODO: Forward pass
  def forward(self, x):
    # Compute weight vectors
    queries = self.W_query(x)
    keys = self.W_key(x)
    values = self.W_value(x)

    # Compute attention scores
    #attn_scores = queries @ keys.T # omega
    attn_scores = queries @ keys.transpose(1, 2) # omega

    # Mask future tokens before normalization so that row sums still sum to 1
    attn_scores.masked_fill(self.mask.bool(), -torch.inf)

    # Compute attention weights by normalizing with softmax
    attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
    attn_weights = self.dropout(attn_weights)

    # Compute context vectors of all inputs
    context_vec = attn_weights @ values
    return context_vec

In [ ]:
torch.manual_seed(123)
context_length = batch.shape[1]
ca = CausalAttention(d_in, d_out, context_length, 0.0)

context_vecs = ca(batch)
print(context_vecs)
print(f"context_vecs.shape: {context_vecs.shape}")